# Virtual Zarr Creation & Performance Benchmarking

## Introduction

To create the data cube in virtualzarr format using the outputs from `gdal` and `cdo`.

### Key Objectives:

1. **Visualize outputs** from CDO and GDAL preprocessing pipelines
2. **Create virtual Zarr stores** from preprocessed netCDF files
3. **Benchmark performance** comparing:
   - NetCDF (original format)
   - Regular Zarr (converted format)
   - Virtual Zarr (reference-based format)
4. **Measure metrics**: Load time, memory usage, file size, and query performance

Virtual Zarrs provide significant advantages:
- 📦 **Minimal disk space**: References files without copying
- ⚡ **Fast loading**: Lazy-loaded metadata and data access
- 💾 **Efficient memory**: Only load required data chunks
- 🔗 **Linked analysis**: Unified access to multiple source files


In [ ]:
# Import Required Libraries (NetCDF -> Zarr workflow only)

# Fix matplotlib backend issue from environment
import os
if "MPLBACKEND" in os.environ:
    del os.environ["MPLBACKEND"]

# Core analysis
import xarray as xr
import numpy as np
import pandas as pd

# Zarr + performance
import zarr
import time
import psutil
import gc

# Optional NetCDF backend import (some environments may not have it)
try:
    import netCDF4  # noqa: F401
except Exception as e:
    print(f"Note: netCDF4 import failed: {e}")

# Visualization
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import rcParams

# File & system ops
import glob
import json
from pathlib import Path
from datetime import datetime

# Environment knobs
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# Warnings & logging
import warnings
warnings.filterwarnings("ignore")

# Configure matplotlib defaults
rcParams["figure.figsize"] = (14, 8)
rcParams["font.size"] = 10
rcParams["axes.labelsize"] = 11
rcParams["axes.titlesize"] = 12
rcParams["xtick.labelsize"] = 9
rcParams["ytick.labelsize"] = 9
rcParams["legend.fontsize"] = 10
rcParams["figure.titlesize"] = 14

print("Imports OK")
print(f"  - xarray: {xr.__version__}")
print(f"  - numpy:  {np.__version__}")
print(f"  - pandas: {pd.__version__}")
print(f"  - zarr:   {zarr.__version__}")

✓ virtualizarr available
  - virtualizarr version: 2.4.0
✓ h5py version: 3.15.1
✓ h5py location: /home/kzakir/dvcube/test/cube-sample/lib/python3.12/site-packages/h5py/__init__.py
All required libraries imported successfully!
  - xarray version: 2025.11.0
  - numpy version: 2.4.2
  - pandas version: 3.0.0
  - zarr version: 3.1.5


## Notebook Configuration

Before running the analysis, ensure you have the following dependencies installed:

```bash
pip install xarray zarr pandas matplotlib psutil netCDF4
```

This notebook uses data outputs from the CDO and GDAL preprocessing pipelines located in:
- `../outputs/cdo_output/` - CDO processed netCDF files
- `../outputs/gdal_output/` - GDAL processed netCDF files
- `../outputs/zarr_outputs/` - Generated Zarr stores (created by this notebook)


## Creating Virtual Zarrs from CDO and GDAL Outputs

This section demonstrates how to create virtual Zarr stores from the netCDF files generated by CDO and GDAL preprocessing pipelines.

Virtual Zarrs allow us to:
- Efficiently access and analyze large multi-file datasets
- Create unified data cubes from scattered netCDF files
- Maintain data integrity without duplicating files
- Enable lazy loading and parallel processing


In [14]:
### 1. Define paths to CDO and GDAL outputs

# Setup directory paths
outputs_dir = "../outputs"
cdo_output_dir, gdal_output_dir = f"{outputs_dir}/cdo_output", f"{outputs_dir}/gdal_output"

# Define subdirectories
paths = {
    'cdo_metref': f"{cdo_output_dir}/metref",
    'cdo_rzsm': f"{cdo_output_dir}/rzsm",
    'gdal_metref': f"{gdal_output_dir}/metref",
    'gdal_sm': f"{gdal_output_dir}/rzsm"
}

# Find all files with corrected glob patterns
cdo_metref_files = sorted(glob.glob(f"{paths['cdo_metref']}/**/*.nc", recursive=True))
cdo_rzsm_files = sorted(glob.glob(f"{paths['cdo_rzsm']}/**/*.nc", recursive=True))
gdal_metref_files = sorted(glob.glob(f"{paths['gdal_metref']}/**/METREF/*.nc", recursive=True))

# GDAL SM files organized by variable folders (var40, var41, var42, var43)
gdal_sm_files = {}
for var_folder in ['var40', 'var41', 'var42', 'var43']:
    var_path = f"{paths['gdal_sm']}/**/{var_folder}/*.nc"
    files = sorted(glob.glob(var_path, recursive=True))
    if files:
        gdal_sm_files[var_folder] = files

# Print summary
print(f"CDO Output: {cdo_output_dir}\nGDAL Output: {gdal_output_dir}")
print(f"\nCDO METREF: {len(cdo_metref_files)} files | CDO RZSM: {len(cdo_rzsm_files)} files")
print(f"GDAL METREF: {len(gdal_metref_files)} files")
print(f"GDAL SM variables found: {list(gdal_sm_files.keys())}")
for var, files in gdal_sm_files.items():
    print(f"  {var}: {len(files)} files")


CDO Output: ../outputs/cdo_output
GDAL Output: ../outputs/gdal_output

CDO METREF: 5 files | CDO RZSM: 5 files
GDAL METREF: 5 files
GDAL SM variables found: ['var40', 'var41', 'var42', 'var43']
  var40: 5 files
  var41: 5 files
  var42: 5 files
  var43: 5 files


In [ ]:
# Configure input NetCDF directory (CDO output example)

store_path = Path("/home/kzakir/dvcube/test/outputs/cdo_output/metref/2010/01").resolve()
assert store_path.exists(), f"Missing input folder: {store_path}"

# Collect NetCDF files
all_files = sorted(glob.glob(str(store_path / "**" / "*.nc"), recursive=True))
print(f"Found {len(all_files)} NetCDF files under {store_path}")
assert len(all_files) > 0, "No .nc files found"

# Preview first file path
print("First file:", all_files[0])

In [ ]:
# Open multiple files with xarray (no virtual zarr)

# If you get backend errors with netCDF4, switch to engine="h5netcdf" here too.
ds_nc = xr.open_mfdataset(
    all_files,
    combine="nested",
    concat_dim="time",
    engine="h5netcdf",
    parallel=False,
)

print(ds_nc)

Found 5 NetCDF files in /home/kzakir/dvcube/test/outputs/cdo_output/metref/2010/01


<xarray.Dataset> Size: 12MB
Dimensions:       (time: 5, lat: 200, lon: 1500)
Coordinates:
  * time          (time) datetime64[ns] 40B 2010-01-01 2010-01-02 ... 2010-01-05
  * lat           (lat) float64 2kB 10.0 10.05 10.1 10.15 ... 19.85 19.9 19.95
  * lon           (lon) float64 12kB -20.0 -19.95 -19.9 ... 54.85 54.9 54.95
Data variables:
    METREF        (time, lat, lon) int32 6MB ManifestArray<shape=(5, 200, 150...
    quality_flag  (time, lat, lon) int32 6MB ManifestArray<shape=(5, 200, 150...
Attributes: (12/29)
    CDI:                        Climate Data Interface version 2.0.4 (https:/...
    Conventions:                CF-1.6
    institution:                IM-PT
    date_created:               2020-11-09T17:30:27Z
    algorithm_version:          1.3.2
    base_algorithm_version:     1.0.3
    ...                         ...
    westernmost_longitude:      80.0
    spatial_resolution:          0.05x 0.05
    geospatial_lat_units:       degrees_north
    geospatial_lon_units:       degrees_east
    netcdf_version_id:          netCDF4
    CDO:                        Climate Data Operators version 2.0.4 (https:/...

## Make Zarr format from the combine files

In [ ]:
# Make a real (materialized) Zarr store from the multi-file NetCDFs using xarray

# If you hit: RuntimeError: Unspecified error in H5DSget_num_scales
# that's an HDF5 dimension-scales metadata issue triggered while opening some NetCDF4/HDF5 files.
# This cell uses a more robust path: open each file "raw" (decode_cf=False) then concat along time.

from pathlib import Path
import shutil
import xarray as xr

assert len(all_files) > 0, "No NetCDF files found in all_files"

zarr_store_path = Path("../outputs/zarr_outputs/metref_combined_xarray.zarr").resolve()
zarr_store_path.parent.mkdir(parents=True, exist_ok=True)
if zarr_store_path.exists():
    shutil.rmtree(zarr_store_path)

t0 = time.time()

# 1) Open each file separately in a minimal way
datasets = []
for fp in all_files:
    try:
        ds_i = xr.open_dataset(
            fp,
            engine="h5netcdf",
            decode_cf=False,
            mask_and_scale=False,
            chunks={},
        )
        datasets.append(ds_i)
    except Exception as e:
        print(f"FAILED to open: {fp}\n  -> {type(e).__name__}: {e}")
        raise

# 2) Concat along time
ds_xr = xr.concat(datasets, dim="time", combine_attrs="override")

# 3) Chunk for Zarr (adjust to your query patterns)
chunk_plan = {"time": 1}
for dim in ["lat", "latitude", "y"]:
    if dim in ds_xr.dims:
        chunk_plan[dim] = 256
for dim in ["lon", "longitude", "x"]:
    if dim in ds_xr.dims:
        chunk_plan[dim] = 256
ds_xr = ds_xr.chunk(chunk_plan)

# 4) Write Zarr (v2)
ds_xr.to_zarr(zarr_store_path, mode="w", consolidated=True)

dt = time.time() - t0
print(f"✓ Wrote Zarr store: {zarr_store_path}")
print(f"  - wall time: {dt:.2f}s")

# 5) Reopen with xr.open_zarr and read a scalar (sanity check)
ds_z = xr.open_zarr(zarr_store_path, consolidated=True)
print(ds_z)

varname = "METREF" if "METREF" in ds_z.data_vars else list(ds_z.data_vars)[0]
print(f"Using variable: {varname}")

scalar_indexers = {"time": 0}
if "lat" in ds_z.dims:
    scalar_indexers["lat"] = 0
if "lon" in ds_z.dims:
    scalar_indexers["lon"] = 0

val = ds_z[varname].isel(**scalar_indexers).load()
print("Scalar (loaded):", float(val.values) if np.ndim(val.values) == 0 else val.values)

NameError: name 'all_files' is not defined